# DeepCFD Extended — Training + Evaluation Pipeline

Single notebook covering **training and testing** of three U-Net variants for 2D steady-state laminar flow prediction, compared against the DeepCFD baseline (Ribeiro et al., 2021). Designed to produce paper-ready figures and tables in the style of the original DeepCFD paper.

| Architecture        | Key mechanism                                  | Reference |
|---------------------|-----------------------------------------------|-----------|
| **UNet-Attention**  | Attention gates on skip connections           | Oktay et al., 2018 |
| **UNet-FNO**        | Fourier Neural Operator bottleneck            | Li et al., 2021    |
| **UNet-Transformer**| Multi-Head Self-Attention bottleneck (ViT)    | Dosovitskiy, 2021  |
| *UNet-Base*         | DeepCFD replica (paper reference)             | Ribeiro et al., 2021 |

All models share identical encoder/decoder backbones. Differences live only in the bottleneck (FNO / MHSA / plain conv) and in the skip pathway (attention-gated or plain), so performance differences reflect the global-mixing mechanism.

**Training modes (each model trained twice):**
- Data-only — weighted MSE (Ux, Uy) + MAE (p), per-channel RMS normalization.
- Data + Physics — continuity + momentum residuals via finite differences, boundary-condition penalties, gradient-norm adaptive balancing (Wang et al., 2021) with ramp-up.

**Improvements over the original notebook:**
1. Bilinear upsampling replaces `MaxUnpool2d` (standard in modern U-Nets, independent decoders).
2. Hard obstacle constraint: `Ux = Uy = p = 0` inside the obstacle, enforced in the forward pass.
3. Pressure gauge fix: subtract per-sample mean on the fluid region (incompressible pressure is defined up to a constant).
4. Filters `[8, 16, 32, 32]` match the DeepCFD paper's best configuration.
5. 70/15/15 train/val/test split with validation-based early stopping and `ReduceLROnPlateau`.
6. Metrics reported on the fluid region only (masked).

---
## Notebook structure
**Part A — Training** (sections 1–12)
Define backbones and three architectures, physics loss, training loop, run all 7 experiments, save checkpoints.

**Part B — Evaluation** (sections 13–21)
Test-set metrics tables, training curves, flow visualization per sample, |U| + p + error grids, data distributions, relative errors, inference speedup, PDE residual maps.

# Part A — Training

## 1. Imports and Setup

In [ ]:
import os
import copy
import time
import math
import pickle
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# ---------- Reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------- Device ----------
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Device: {device}  |  seed: {SEED}")


## 2. Shared Backbone Blocks

All three architectures share the same `EncoderBlock`, `DecoderBlock`, `AttentionGate` / `DecoderBlockAttn`, and `HardConstraintHead`. They differ only in the bottleneck (and in whether the skip decoders use attention).

In [ ]:
class ConvBlock(nn.Module):
    """Conv -> ReLU. Padding keeps spatial dims constant."""
    def __init__(self, in_ch, out_ch, kernel_size=5):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size, padding=kernel_size // 2)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv(x))


class EncoderBlock(nn.Module):
    """Encoder: 2x Conv -> MaxPool. Returns pooled feature and skip."""
    def __init__(self, in_ch, out_ch, kernel_size=5):
        super().__init__()
        self.conv1 = ConvBlock(in_ch, out_ch, kernel_size)
        self.conv2 = ConvBlock(out_ch, out_ch, kernel_size)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        skip = x
        x = self.pool(x)
        return x, skip


class DecoderBlock(nn.Module):
    """Decoder: bilinear upsample -> concat skip -> 2x Conv."""
    def __init__(self, in_ch, skip_ch, out_ch, kernel_size=5):
        super().__init__()
        self.conv1 = ConvBlock(in_ch + skip_ch, out_ch, kernel_size)
        self.conv2 = ConvBlock(out_ch, out_ch, kernel_size)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        x = torch.cat([skip, x], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x


class AttentionGate(nn.Module):
    """Attention Gate (Oktay et al., 2018) — soft attention on skip connections.

    Produces a coefficient map alpha in [0, 1] used to re-weight the skip.
    """
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g_up = F.interpolate(g, size=x.shape[-2:], mode='bilinear', align_corners=False)
        psi = self.relu(self.W_g(g_up) + self.W_x(x))
        alpha = self.psi(psi)
        return x * alpha


class DecoderBlockAttn(nn.Module):
    """Decoder with Attention Gate on the skip connection."""
    def __init__(self, in_ch, skip_ch, out_ch, kernel_size=5):
        super().__init__()
        self.attn = AttentionGate(F_g=in_ch, F_l=skip_ch, F_int=max(1, skip_ch // 2))
        self.conv1 = ConvBlock(in_ch + skip_ch, out_ch, kernel_size)
        self.conv2 = ConvBlock(out_ch, out_ch, kernel_size)

    def forward(self, x, skip):
        skip_att = self.attn(g=x, x=skip)
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        x = torch.cat([skip_att, x], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x


class HardConstraintHead(nn.Module):
    """Applies:
      (a) obstacle mask (Ux = Uy = p = 0 inside the obstacle)
      (b) per-sample zero-mean pressure (incompressible gauge fix)

    The flow-region channel (input[:, 1]) uses the convention:
      0 = obstacle, 1 = fluid, 2 = wall, 3 = inlet, 4 = outlet.
    """
    def __init__(self, enforce_obstacle=True, zero_mean_p=True):
        super().__init__()
        self.enforce_obstacle = enforce_obstacle
        self.zero_mean_p = zero_mean_p

    def forward(self, y_hat, x_in):
        out = y_hat
        fluid_mask = None
        if self.enforce_obstacle:
            region = x_in[:, 1:2]
            fluid_mask = (region > 0.5).float()
            out = out * fluid_mask

        if self.zero_mean_p:
            p = out[:, 2:3]
            if fluid_mask is not None:
                denom = fluid_mask.sum(dim=(1, 2, 3), keepdim=True).clamp_min(1.0)
                p_mean = (p * fluid_mask).sum(dim=(1, 2, 3), keepdim=True) / denom
            else:
                p_mean = p.mean(dim=(1, 2, 3), keepdim=True)
            out = torch.cat([out[:, 0:2], p - p_mean], dim=1)
            if fluid_mask is not None:
                out = out * fluid_mask

        return out


print("Backbone blocks defined.")


## 3. Architecture 1 — UNet-Attention

Attention U-Net (Oktay et al., 2018) with the 3-decoder configuration, which is the DeepCFD paper's best variant. The attention gates let the decoder selectively weight skip features — useful when the obstacle occupies a small fraction of the domain.

`UNet-Base` (no attention) is included as a sanity replica of DeepCFD.

In [ ]:
class UNetBase(nn.Module):
    """Plain U-Net (DeepCFD replica) for sanity checks. Same backbone, no attention."""
    name = "UNet-Base"

    def __init__(self, in_channels=3, out_channels=3, kernel_size=5,
                 filters=(8, 16, 32, 32), enforce_obstacle=True, zero_mean_p=True):
        super().__init__()
        f = list(filters)

        self.enc1 = EncoderBlock(in_channels, f[0], kernel_size)
        self.enc2 = EncoderBlock(f[0], f[1], kernel_size)
        self.enc3 = EncoderBlock(f[1], f[2], kernel_size)
        self.enc4 = EncoderBlock(f[2], f[3], kernel_size)

        self.bottleneck = nn.Sequential(
            ConvBlock(f[3], f[3], kernel_size),
            ConvBlock(f[3], f[3], kernel_size),
        )

        def build_decoder():
            return nn.ModuleList([
                DecoderBlock(f[3], f[3], f[2], kernel_size),
                DecoderBlock(f[2], f[2], f[1], kernel_size),
                DecoderBlock(f[1], f[1], f[0], kernel_size),
                DecoderBlock(f[0], f[0], f[0], kernel_size),
                nn.Conv2d(f[0], 1, kernel_size=1),
            ])
        self.decoders = nn.ModuleList([build_decoder() for _ in range(out_channels)])
        self.head = HardConstraintHead(enforce_obstacle, zero_mean_p)

    def forward(self, x):
        x_in = x
        x1, s1 = self.enc1(x)
        x2, s2 = self.enc2(x1)
        x3, s3 = self.enc3(x2)
        x4, s4 = self.enc4(x3)
        z = self.bottleneck(x4)
        outs = []
        for dec in self.decoders:
            d = dec[0](z, s4)
            d = dec[1](d, s3)
            d = dec[2](d, s2)
            d = dec[3](d, s1)
            d = dec[4](d)
            outs.append(d)
        y = torch.cat(outs, dim=1)
        return self.head(y, x_in)


class UNetAttention(nn.Module):
    """U-Net with Attention Gates on skip connections + 3 separate decoders.

    Architecture 1 in the comparison. Reference: Oktay et al., 2018.
    """
    name = "UNet-Attention"

    def __init__(self, in_channels=3, out_channels=3, kernel_size=5,
                 filters=(8, 16, 32, 32), enforce_obstacle=True, zero_mean_p=True):
        super().__init__()
        f = list(filters)

        self.enc1 = EncoderBlock(in_channels, f[0], kernel_size)
        self.enc2 = EncoderBlock(f[0], f[1], kernel_size)
        self.enc3 = EncoderBlock(f[1], f[2], kernel_size)
        self.enc4 = EncoderBlock(f[2], f[3], kernel_size)

        self.bottleneck = nn.Sequential(
            ConvBlock(f[3], f[3], kernel_size),
            ConvBlock(f[3], f[3], kernel_size),
        )

        def build_decoder():
            return nn.ModuleList([
                DecoderBlockAttn(f[3], f[3], f[2], kernel_size),
                DecoderBlockAttn(f[2], f[2], f[1], kernel_size),
                DecoderBlockAttn(f[1], f[1], f[0], kernel_size),
                DecoderBlockAttn(f[0], f[0], f[0], kernel_size),
                nn.Conv2d(f[0], 1, kernel_size=1),
            ])
        self.decoders = nn.ModuleList([build_decoder() for _ in range(out_channels)])
        self.head = HardConstraintHead(enforce_obstacle, zero_mean_p)

    def forward(self, x):
        x_in = x
        x1, s1 = self.enc1(x)
        x2, s2 = self.enc2(x1)
        x3, s3 = self.enc3(x2)
        x4, s4 = self.enc4(x3)
        z = self.bottleneck(x4)
        outs = []
        for dec in self.decoders:
            d = dec[0](z, s4)
            d = dec[1](d, s3)
            d = dec[2](d, s2)
            d = dec[3](d, s1)
            d = dec[4](d)
            outs.append(d)
        y = torch.cat(outs, dim=1)
        return self.head(y, x_in)


print("UNet-Base and UNet-Attention defined.")


## 4. Architecture 2 — UNet-FNO

**Fourier Neural Operator** bottleneck (Li et al., 2021): FFT → truncate modes → complex matmul → iFFT. Each pixel at the bottleneck has a full-domain receptive field in one layer, so global mixing is efficient.

Spectral bias favors smooth, large-scale structure — a good match for steady laminar flows. We keep only a handful of low modes because the bottleneck resolution is already small (~10 × 4 after four MaxPool(2) layers).

In [ ]:
class SpectralConv2d(nn.Module):
    """2D spectral convolution: FFT -> truncate modes -> complex matmul -> iFFT.

    Implements the integral-operator layer from the Fourier Neural Operator
    (Li et al., 2021). Keeps only the lowest ``modes1 x modes2`` Fourier modes.
    """
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2

        scale = 1.0 / (in_channels * out_channels)
        # Two weight tensors: one for the positive-frequency block and
        # one for the conjugate block (exploits FFT symmetry for real input).
        self.weights1 = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat)
        )
        self.weights2 = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat)
        )

    @staticmethod
    def compl_mul2d(a, b):
        # a: (B, C_in, H, W)  b: (C_in, C_out, H, W)  ->  (B, C_out, H, W)
        return torch.einsum("bixy,ioxy->boxy", a, b)

    def forward(self, x):
        B, C, H, W = x.shape
        x_ft = torch.fft.rfft2(x, norm='ortho')            # (B, C, H, W//2+1)
        m1 = min(self.modes1, H // 2)
        m2 = min(self.modes2, W // 2 + 1)

        out_ft = torch.zeros(B, self.out_channels, H, W // 2 + 1,
                             dtype=torch.cfloat, device=x.device)
        # Positive-frequency block
        out_ft[:, :, :m1, :m2] = self.compl_mul2d(
            x_ft[:, :, :m1, :m2], self.weights1[:, :, :m1, :m2]
        )
        # Conjugate block (high-x-frequency)
        out_ft[:, :, -m1:, :m2] = self.compl_mul2d(
            x_ft[:, :, -m1:, :m2], self.weights2[:, :, :m1, :m2]
        )
        return torch.fft.irfft2(out_ft, s=(H, W), norm='ortho')


class FNOBlock(nn.Module):
    """One FNO layer: SpectralConv + 1x1 Conv (linear skip), activated by GELU."""
    def __init__(self, channels, modes1, modes2, activation=True):
        super().__init__()
        self.spec = SpectralConv2d(channels, channels, modes1, modes2)
        self.w = nn.Conv2d(channels, channels, 1)
        self.activation = activation

    def forward(self, x):
        y = self.spec(x) + self.w(x)
        if self.activation:
            y = F.gelu(y)
        return y


class FNOBottleneck(nn.Module):
    """Stack of FNO layers used as the U-Net bottleneck."""
    def __init__(self, channels, modes1=4, modes2=3, num_layers=4):
        super().__init__()
        layers = []
        for i in range(num_layers):
            layers.append(FNOBlock(channels, modes1, modes2, activation=(i < num_layers - 1)))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class UNetFNO(nn.Module):
    """U-Net with Fourier Neural Operator bottleneck and attention-gated skips.

    Architecture 2 in the comparison. The FNO bottleneck mixes information
    globally in Fourier space, so each pixel at the bottleneck has full
    receptive field in one layer. Complementary to the local CNN encoder/decoder.
    """
    name = "UNet-FNO"

    def __init__(self, in_channels=3, out_channels=3, kernel_size=5,
                 filters=(8, 16, 32, 32), fno_modes=(4, 3), fno_layers=4,
                 enforce_obstacle=True, zero_mean_p=True):
        super().__init__()
        f = list(filters)

        self.enc1 = EncoderBlock(in_channels, f[0], kernel_size)
        self.enc2 = EncoderBlock(f[0], f[1], kernel_size)
        self.enc3 = EncoderBlock(f[1], f[2], kernel_size)
        self.enc4 = EncoderBlock(f[2], f[3], kernel_size)

        self.bottleneck = FNOBottleneck(f[3], modes1=fno_modes[0], modes2=fno_modes[1],
                                         num_layers=fno_layers)

        def build_decoder():
            return nn.ModuleList([
                DecoderBlockAttn(f[3], f[3], f[2], kernel_size),
                DecoderBlockAttn(f[2], f[2], f[1], kernel_size),
                DecoderBlockAttn(f[1], f[1], f[0], kernel_size),
                DecoderBlockAttn(f[0], f[0], f[0], kernel_size),
                nn.Conv2d(f[0], 1, kernel_size=1),
            ])
        self.decoders = nn.ModuleList([build_decoder() for _ in range(out_channels)])
        self.head = HardConstraintHead(enforce_obstacle, zero_mean_p)

    def forward(self, x):
        x_in = x
        x1, s1 = self.enc1(x)
        x2, s2 = self.enc2(x1)
        x3, s3 = self.enc3(x2)
        x4, s4 = self.enc4(x3)
        z = self.bottleneck(x4)
        outs = []
        for dec in self.decoders:
            d = dec[0](z, s4)
            d = dec[1](d, s3)
            d = dec[2](d, s2)
            d = dec[3](d, s1)
            d = dec[4](d)
            outs.append(d)
        y = torch.cat(outs, dim=1)
        return self.head(y, x_in)


print("UNet-FNO defined.")


## 5. Architecture 3 — UNet-Transformer

**ViT-style Multi-Head Self-Attention** bottleneck. The bottleneck feature map is tokenized, enriched with learned positional embeddings, and processed by Transformer encoder blocks.

Difference from FNO: attention weights are data-dependent (each token chooses who to attend to based on content), complementing FNO's fixed spectral mixing.

In [ ]:
class TransformerBlock(nn.Module):
    """Pre-norm Transformer encoder block: MHSA + MLP with residual connections."""
    def __init__(self, embed_dim, num_heads=4, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout,
                                          batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        xn = self.norm1(x)
        a, _ = self.attn(xn, xn, xn, need_weights=False)
        x = x + a
        x = x + self.mlp(self.norm2(x))
        return x


class TransformerBottleneck(nn.Module):
    """Flatten -> 1x1 proj -> Transformer stack -> unflatten. ViT-style."""
    def __init__(self, channels, bottleneck_hw, embed_dim=128, num_heads=4,
                 num_layers=2, dropout=0.0):
        super().__init__()
        H, W = bottleneck_hw
        self.proj_in = nn.Conv2d(channels, embed_dim, 1)
        self.proj_out = nn.Linear(embed_dim, channels)
        self.pos_embed = nn.Parameter(torch.zeros(1, H * W, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(channels)
        self.hw = (H, W)

    def forward(self, x):
        B, C, H, W = x.shape
        assert (H, W) == self.hw, f"Expected bottleneck {self.hw}, got {(H, W)}"
        t = self.proj_in(x).flatten(2).transpose(1, 2)  # [B, N, D]
        t = t + self.pos_embed
        for blk in self.blocks:
            t = blk(t)
        t = self.proj_out(t)
        t = self.norm(t)
        return t.transpose(1, 2).reshape(B, C, H, W)


class UNetTransformer(nn.Module):
    """U-Net with ViT-style multi-head self-attention bottleneck + attention skips.

    Architecture 3 in the comparison. Unlike FNO (fixed spectral filters after
    training), self-attention weights are *data-dependent*: each token decides
    which other tokens to attend to based on its content. This can capture
    non-local content-specific interactions (e.g., wake-obstacle coupling).
    """
    name = "UNet-Transformer"

    def __init__(self, in_channels=3, out_channels=3, kernel_size=5,
                 filters=(8, 16, 32, 32), input_hw=(172, 79),
                 embed_dim=128, num_heads=4, num_transformer_layers=2,
                 enforce_obstacle=True, zero_mean_p=True):
        super().__init__()
        f = list(filters)

        self.enc1 = EncoderBlock(in_channels, f[0], kernel_size)
        self.enc2 = EncoderBlock(f[0], f[1], kernel_size)
        self.enc3 = EncoderBlock(f[1], f[2], kernel_size)
        self.enc4 = EncoderBlock(f[2], f[3], kernel_size)

        # After 4x MaxPool(2) the bottleneck is (input_hw[0]//16, input_hw[1]//16)
        bh, bw = input_hw[0] // 16, input_hw[1] // 16
        self.bottleneck = TransformerBottleneck(
            f[3], bottleneck_hw=(bh, bw),
            embed_dim=embed_dim, num_heads=num_heads, num_layers=num_transformer_layers,
        )

        def build_decoder():
            return nn.ModuleList([
                DecoderBlockAttn(f[3], f[3], f[2], kernel_size),
                DecoderBlockAttn(f[2], f[2], f[1], kernel_size),
                DecoderBlockAttn(f[1], f[1], f[0], kernel_size),
                DecoderBlockAttn(f[0], f[0], f[0], kernel_size),
                nn.Conv2d(f[0], 1, kernel_size=1),
            ])
        self.decoders = nn.ModuleList([build_decoder() for _ in range(out_channels)])
        self.head = HardConstraintHead(enforce_obstacle, zero_mean_p)

    def forward(self, x):
        x_in = x
        x1, s1 = self.enc1(x)
        x2, s2 = self.enc2(x1)
        x3, s3 = self.enc3(x2)
        x4, s4 = self.enc4(x3)
        z = self.bottleneck(x4)
        outs = []
        for dec in self.decoders:
            d = dec[0](z, s4)
            d = dec[1](d, s3)
            d = dec[2](d, s2)
            d = dec[3](d, s1)
            d = dec[4](d)
            outs.append(d)
        y = torch.cat(outs, dim=1)
        return self.head(y, x_in)


print("UNet-Transformer defined.")


## 6. Parameter Budget

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


_models_info = {
    'UNet-Base':        UNetBase(filters=(8, 16, 32, 32)),
    'UNet-Attention':   UNetAttention(filters=(8, 16, 32, 32)),
    'UNet-FNO':         UNetFNO(filters=(8, 16, 32, 32), fno_modes=(4, 3), fno_layers=4),
    'UNet-Transformer': UNetTransformer(filters=(8, 16, 32, 32),
                                        embed_dim=128, num_heads=4, num_transformer_layers=2),
}

descriptions = {
    'UNet-Base':        'DeepCFD replica (paper reference)',
    'UNet-Attention':   'attention-gated skips',
    'UNet-FNO':         'FNO bottleneck (spectral global mixing)',
    'UNet-Transformer': 'MHSA bottleneck (attention global mixing)',
}

print(f"{'Architecture':<22}{'Params':>12}  Description")
print('-' * 72)
for name, m in _models_info.items():
    print(f"{name:<22}{count_params(m):>12,}  {descriptions[name]}")

del _models_info


## 7. Physics-Informed Loss

Residuals of the 2D steady-state incompressible Navier–Stokes equations, evaluated with second-order central finite differences on the CFD grid.

**Continuity:** $\nabla \cdot \mathbf{u} = 0$

**Momentum:** $(\mathbf{u} \cdot \nabla) \mathbf{u} = -\nabla p + \nu \nabla^2 \mathbf{u}$

Grid: $n_x = 172$ along the stream-wise direction (H in NCHW), $n_y = 79$ perpendicular (W). Spacings $dx = 0.260/171$, $dy = 0.120/78$, viscosity $\nu = 10^{-4}$, inlet velocity $U_{\text{inlet}} = 0.1$.

PDE residuals are evaluated on the interior fluid region only (fluid minus one-cell border). Boundary terms penalize no-slip (obstacle + walls), inlet, and outlet zero-gradient.

**Loss balancing:** $\mathcal{L} = \mathcal{L}_{\text{data}} + \lambda_{\text{eff}} \mathcal{L}_{\text{physics}}$ with $\lambda_{\text{eff}} = \lambda_{\text{base}} \cdot r(e) \cdot \hat\lambda(e)$: $r(e)$ is a linear ramp, $\hat\lambda(e)$ is the gradient-norm adaptive scalar (Wang et al., 2021).

In [ ]:
class PhysicsInformedLoss(nn.Module):
    """Residuals of 2D steady-state incompressible Navier-Stokes via finite differences.

    Input tensor layout [B, C, H, W]: H=172 stream-wise (x), W=79 perpendicular (y).
    Kernel ``k_dx`` has shape (1, 1, 3, 1) so it operates along the H axis (x).
    BC channel convention: 0=obstacle, 1=fluid, 2=wall, 3=inlet, 4=outlet.
    """
    def __init__(self, nx=172, ny=79, lx=0.260, ly=0.120, nu=1e-4, u_inlet=0.1):
        super().__init__()
        self.nu = nu
        self.u_inlet = u_inlet
        dx = lx / (nx - 1)
        dy = ly / (ny - 1)
        self.dx = dx
        self.dy = dy

        # First-derivative kernels (central differences, O(dx^2))
        k_dx = torch.zeros(1, 1, 3, 1)
        k_dx[0, 0, 0, 0] = -1.0 / (2.0 * dx)
        k_dx[0, 0, 2, 0] =  1.0 / (2.0 * dx)
        self.register_buffer('kernel_dx', k_dx)

        k_dy = torch.zeros(1, 1, 1, 3)
        k_dy[0, 0, 0, 0] = -1.0 / (2.0 * dy)
        k_dy[0, 0, 0, 2] =  1.0 / (2.0 * dy)
        self.register_buffer('kernel_dy', k_dy)

        # Second-derivative kernels (central, O(dx^2))
        k_d2x = torch.zeros(1, 1, 3, 1)
        k_d2x[0, 0, 0, 0] =  1.0 / dx**2
        k_d2x[0, 0, 1, 0] = -2.0 / dx**2
        k_d2x[0, 0, 2, 0] =  1.0 / dx**2
        self.register_buffer('kernel_d2x', k_d2x)

        k_d2y = torch.zeros(1, 1, 1, 3)
        k_d2y[0, 0, 0, 0] =  1.0 / dy**2
        k_d2y[0, 0, 0, 1] = -2.0 / dy**2
        k_d2y[0, 0, 0, 2] =  1.0 / dy**2
        self.register_buffer('kernel_d2y', k_d2y)

    def _deriv(self, f, kernel):
        kh, kw = kernel.shape[-2], kernel.shape[-1]
        f_pad = F.pad(f, (kw // 2, kw // 2, kh // 2, kh // 2), mode='replicate')
        return F.conv2d(f_pad, kernel)

    def ddx(self, f):   return self._deriv(f, self.kernel_dx)
    def ddy(self, f):   return self._deriv(f, self.kernel_dy)
    def d2dx2(self, f): return self._deriv(f, self.kernel_d2x)
    def d2dy2(self, f): return self._deriv(f, self.kernel_d2y)

    def region_masks(self, x_in):
        r = x_in[:, 1:2]
        fluid    = (r > 0.5).float()
        obstacle = (r < 0.5).float()
        wall     = ((r > 1.5) & (r < 2.5)).float()
        inlet    = ((r > 2.5) & (r < 3.5)).float()
        outlet   = (r > 3.5).float()
        noslip   = ((obstacle + wall) > 0.5).float()

        # Interior: fluid minus one-cell border (avoids stencil reaching outside)
        interior = fluid.clone()
        interior[:, :, :1, :] = 0
        interior[:, :, -1:, :] = 0
        interior[:, :, :, :1] = 0
        interior[:, :, :, -1:] = 0
        return dict(fluid=fluid, interior=interior, obstacle=obstacle,
                    wall=wall, noslip=noslip, inlet=inlet, outlet=outlet)

    @staticmethod
    def _masked_mean(x, mask):
        return (x * mask).sum() / mask.sum().clamp_min(1.0)

    def continuity(self, ux, uy, mask):
        div = self.ddx(ux) + self.ddy(uy)
        return self._masked_mean(div ** 2, mask)

    def momentum(self, ux, uy, p, mask):
        dux_dx = self.ddx(ux); dux_dy = self.ddy(ux)
        duy_dx = self.ddx(uy); duy_dy = self.ddy(uy)
        dp_dx  = self.ddx(p);  dp_dy  = self.ddy(p)
        lap_ux = self.d2dx2(ux) + self.d2dy2(ux)
        lap_uy = self.d2dx2(uy) + self.d2dy2(uy)
        res_x = ux * dux_dx + uy * dux_dy + dp_dx - self.nu * lap_ux
        res_y = ux * duy_dx + uy * duy_dy + dp_dy - self.nu * lap_uy
        return self._masked_mean(res_x ** 2, mask) + self._masked_mean(res_y ** 2, mask)

    def boundary(self, ux, uy, masks):
        loss = ux.new_tensor(0.0)
        noslip = masks['noslip']
        if noslip.sum() > 0:
            loss = loss + self._masked_mean(ux ** 2, noslip) + self._masked_mean(uy ** 2, noslip)
        inlet = masks['inlet']
        if inlet.sum() > 0:
            loss = loss + self._masked_mean((ux - self.u_inlet) ** 2, inlet)
            loss = loss + self._masked_mean(uy ** 2, inlet)
        outlet = masks['outlet']
        if outlet.sum() > 0:
            loss = loss + self._masked_mean(self.ddx(ux) ** 2, outlet)
        return loss

    def forward(self, y_hat, x_in, weights=None):
        if weights is None:
            weights = dict(continuity=1.0, momentum=0.1, boundary=1.0)
        ux = y_hat[:, 0:1]; uy = y_hat[:, 1:2]; p = y_hat[:, 2:3]
        m = self.region_masks(x_in)
        l_cont = self.continuity(ux, uy, m['interior'])
        l_mom  = self.momentum(ux, uy, p, m['interior'])
        l_bc   = self.boundary(ux, uy, m)
        total  = (weights['continuity'] * l_cont
                  + weights['momentum']   * l_mom
                  + weights['boundary']   * l_bc)
        return dict(continuity=l_cont, momentum=l_mom, boundary=l_bc, total_physics=total)


print("PhysicsInformedLoss defined.")
print(f"  dx = {0.260/171:.6e} m,  dy = {0.120/78:.6e} m")
print(f"  nu = 1e-4 m^2/s,  U_inlet = 0.1 m/s")


## 8. Data Loading and Splits

70/15/15 train/validation/test split. Validation drives early stopping and LR scheduling; the test set is held out for the evaluation in Part B.

In [ ]:
x = pickle.load(open('./dataset/dataX.pkl', 'rb'))
y = pickle.load(open('./dataset/dataY.pkl', 'rb'))

print(f"Input  shape: {x.shape}")
print(f"Output shape: {y.shape}")

x = torch.as_tensor(x, dtype=torch.float32)
y = torch.as_tensor(y, dtype=torch.float32)

# Channel weights (RMS per output channel), used to normalize the per-channel
# weighted data loss — same idea as the original DeepCFD code.
B, _, H, W = y.shape
channel_weights = torch.sqrt(
    torch.mean(y.permute(0, 2, 3, 1).reshape(B * H * W, 3) ** 2, dim=0)
).view(1, -1, 1, 1).to(device)
print(f"Channel weights (Ux, Uy, p): {channel_weights.cpu().numpy().flatten()}")

# Reproducible shuffle + 70/15/15 split
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(x), generator=g)
x, y = x[perm], y[perm]

n = len(x)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
train_x, train_y = x[:n_train],              y[:n_train]
val_x,   val_y   = x[n_train:n_train+n_val], y[n_train:n_train+n_val]
test_x,  test_y  = x[n_train+n_val:],        y[n_train+n_val:]

print(f"Train: {len(train_x)} | Val: {len(val_x)} | Test: {len(test_x)}")

batch_size = 32
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(val_x,   val_y),   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(TensorDataset(test_x,  test_y),  batch_size=batch_size, shuffle=False)

# Persist the split so the testing notebook can load the exact same test set.
torch.save({'test_x': test_x, 'test_y': test_y,
            'val_x':  val_x,  'val_y':  val_y,
            'channel_weights': channel_weights.cpu()},
           'data_splits.pt')
print("Saved data_splits.pt")


## 9. Training Utilities

- `data_loss_fn`: DeepCFD weighted loss, masked to the fluid region.
- `gradnorm_lambda`: gradient-norm adaptive scalar (detached).
- `physics_ramp`: linear ramp for the physics weight.
- `train_one_epoch` / `evaluate`: standard loops with per-channel breakdown.

In [ ]:
def data_loss_fn(output, target, channel_weights, region_mask=None):
    """Paper weighted loss: MSE on Ux, Uy; MAE on p. Per-channel RMS normalization."""
    l_ux = ((output[:, 0] - target[:, 0]) ** 2).unsqueeze(1)
    l_uy = ((output[:, 1] - target[:, 1]) ** 2).unsqueeze(1)
    l_p  = torch.abs(output[:, 2] - target[:, 2]).unsqueeze(1)
    per_channel = torch.cat([l_ux, l_uy, l_p], dim=1) / channel_weights

    if region_mask is not None:
        mask = region_mask.expand_as(per_channel)
        return (per_channel * mask).sum() / mask.sum().clamp_min(1.0)
    return per_channel.mean()


def masked_mse_breakdown(output, target, region_mask=None):
    """MSE total and per-channel, masked to the fluid region if provided."""
    sq = (output - target) ** 2
    if region_mask is not None:
        mask = region_mask.expand_as(sq)
        denom = mask.sum().clamp_min(1.0)
        mse_total = (sq * mask).sum().item() / denom.item()
        per = []
        d = region_mask.sum().clamp_min(1.0)
        for c in range(sq.shape[1]):
            per.append(((sq[:, c:c+1] * region_mask).sum() / d).item())
        return mse_total, per[0], per[1], per[2]
    return (sq.mean().item(), sq[:, 0].mean().item(),
            sq[:, 1].mean().item(), sq[:, 2].mean().item())


def physics_ramp(epoch, ramp_start=20, ramp_end=80):
    """Linear ramp from 0 -> 1 for the physics-loss weight."""
    if epoch < ramp_start:
        return 0.0
    if epoch > ramp_end:
        return 1.0
    return (epoch - ramp_start) / (ramp_end - ramp_start)


def gradnorm_lambda(l_data, l_phys, output, eps=1e-8, clamp_range=(1e-3, 10.0)):
    """Gradient-norm balancing (Wang et al., 2021) computed on the output tensor.
    Detached so it does not enter the compute graph of the main backward pass.
    """
    g_d = torch.autograd.grad(l_data, output, retain_graph=True, create_graph=False)[0]
    g_p = torch.autograd.grad(l_phys, output, retain_graph=True, create_graph=False)[0]
    lam = (g_d.abs().mean() / (g_p.abs().mean() + eps)).detach()
    return torch.clamp(lam, *clamp_range)


def train_one_epoch(model, loader, optimizer, channel_weights,
                    physics_fn=None, lambda_phys=0.01, epoch=0,
                    adaptive=True, grad_clip=1.0):
    model.train()
    ramp = physics_ramp(epoch) if physics_fn is not None else 0.0
    totals = dict(loss=0.0, loss_data=0.0, loss_phys=0.0, lam_eff=0.0,
                  mse=0.0, mse_ux=0.0, mse_uy=0.0, mse_p=0.0)
    N = 0

    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model(bx)

        fluid_mask = (bx[:, 1:2] > 0.5).float()
        l_data = data_loss_fn(out, by, channel_weights, region_mask=fluid_mask)

        l_phys = torch.zeros((), device=device)
        lam_dyn = torch.ones((), device=device)
        if physics_fn is not None and ramp > 0:
            phys = physics_fn(out, bx)
            l_phys = phys['total_physics']
            if adaptive:
                lam_dyn = gradnorm_lambda(l_data, l_phys, out)

        lam_eff = lambda_phys * ramp * lam_dyn
        loss = l_data + lam_eff * l_phys
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        n = len(bx); N += n
        totals['loss']      += loss.item() * n
        totals['loss_data'] += l_data.item() * n
        totals['loss_phys'] += float(l_phys) * n
        totals['lam_eff']   += float(lam_eff) * n
        with torch.no_grad():
            mse_t, mse_u, mse_v, mse_p = masked_mse_breakdown(out, by, fluid_mask)
            totals['mse']    += mse_t * n
            totals['mse_ux'] += mse_u * n
            totals['mse_uy'] += mse_v * n
            totals['mse_p']  += mse_p * n

    for k in totals:
        totals[k] /= N
    return totals


@torch.no_grad()
def evaluate(model, loader, channel_weights, physics_fn=None):
    model.eval()
    totals = dict(loss=0.0, loss_phys=0.0,
                  mse=0.0, mse_ux=0.0, mse_uy=0.0, mse_p=0.0)
    N = 0
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        out = model(bx)
        fluid_mask = (bx[:, 1:2] > 0.5).float()

        l_data = data_loss_fn(out, by, channel_weights, region_mask=fluid_mask)
        totals['loss'] += l_data.item() * len(bx)

        if physics_fn is not None:
            totals['loss_phys'] += physics_fn(out, bx)['total_physics'].item() * len(bx)

        mse_t, mse_u, mse_v, mse_p = masked_mse_breakdown(out, by, fluid_mask)
        totals['mse']    += mse_t * len(bx)
        totals['mse_ux'] += mse_u * len(bx)
        totals['mse_uy'] += mse_v * len(bx)
        totals['mse_p']  += mse_p * len(bx)
        N += len(bx)

    for k in totals:
        totals[k] /= N
    return totals


print("Training utilities defined.")


## 10. Experiment Runner

AdamW + ReduceLROnPlateau + early stopping. Best-on-validation checkpoint is restored before returning.

In [ ]:
def run_experiment(model, tag, train_loader, val_loader, channel_weights,
                   use_physics=False, epochs=300, lr=1e-3, patience=40,
                   lambda_phys=0.01, phys_weights=None, adaptive=True,
                   verbose_every=25):
    """Trains a model with optional physics loss. Returns a dict with the best model."""
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=15)

    physics_fn = None
    if use_physics:
        if phys_weights is None:
            phys_weights = dict(continuity=1.0, momentum=0.1, boundary=1.0)
        base_phys = PhysicsInformedLoss().to(device)
        physics_fn = lambda y_hat, x_in: base_phys(y_hat, x_in, phys_weights)

    suffix = ' + Physics' if use_physics else ' (Data only)'
    print('=' * 68)
    print(f"Training {tag}{suffix}")
    print(f"  Params: {count_params(model):,}")
    if use_physics:
        print(f"  lambda_phys(base)={lambda_phys}  adaptive={adaptive}  weights={phys_weights}")
    print('=' * 68)

    history = {'train': [], 'val': []}
    best_val = float('inf')
    best_state = None
    patience_left = patience
    t0 = time.time()

    for ep in range(1, epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer, channel_weights,
                             physics_fn=physics_fn, lambda_phys=lambda_phys,
                             epoch=ep, adaptive=adaptive)
        va = evaluate(model, val_loader, channel_weights, physics_fn=physics_fn)
        scheduler.step(va['loss'])

        history['train'].append(tr)
        history['val'].append(va)

        if ep == 1 or ep % verbose_every == 0:
            msg = (f"  Ep {ep:4d} | lr={optimizer.param_groups[0]['lr']:.1e} "
                   f"| train {tr['loss']:.4e} | val {va['loss']:.4e} "
                   f"| val MSE {va['mse']:.4e}")
            if use_physics:
                msg += f" | phys {va['loss_phys']:.3e} | lam_eff {tr['lam_eff']:.2e}"
            print(msg)

        if va['loss'] < best_val - 1e-9:
            best_val = va['loss']
            best_state = copy.deepcopy(model.state_dict())
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"  Early stop at epoch {ep}")
                break

    elapsed = time.time() - t0
    model.load_state_dict(best_state)
    print(f"  Done in {elapsed:.1f}s | best val loss {best_val:.4e}")

    return dict(tag=f"{tag}{suffix}", model=model, history=history,
                best_val_loss=best_val, time=elapsed, use_physics=use_physics)


print("Experiment runner ready.")


## 11. Run Experiments

Trains 3 architectures × 2 modes + 1 reference = 7 models. On a recent GPU, ~5–10 min per model at 300 epochs. Set `EPOCHS` lower for a quick smoke test.

In [ ]:
EPOCHS = 300      # paper runs up to 1000; 300 with early stop is usually enough
PATIENCE = 40
LR = 1e-3
LAMBDA_PHYS = 0.01

FILTERS = (8, 16, 32, 32)
INPUT_HW = (172, 79)


def make_models():
    return {
        'base':  UNetBase(filters=FILTERS),
        'attn':  UNetAttention(filters=FILTERS),
        'fno':   UNetFNO(filters=FILTERS, fno_modes=(4, 3), fno_layers=4),
        'trans': UNetTransformer(filters=FILTERS, input_hw=INPUT_HW,
                                 embed_dim=128, num_heads=4, num_transformer_layers=2),
    }


results = {}

# --- Data-only reference (DeepCFD replica) ---
m = make_models()
results['base_data'] = run_experiment(
    m['base'], 'UNet-Base', train_loader, val_loader, channel_weights,
    use_physics=False, epochs=EPOCHS, lr=LR, patience=PATIENCE)

# --- Three architectures, data-only ---
results['attn_data'] = run_experiment(
    m['attn'], 'UNet-Attention', train_loader, val_loader, channel_weights,
    use_physics=False, epochs=EPOCHS, lr=LR, patience=PATIENCE)

results['fno_data'] = run_experiment(
    m['fno'], 'UNet-FNO', train_loader, val_loader, channel_weights,
    use_physics=False, epochs=EPOCHS, lr=LR, patience=PATIENCE)

results['trans_data'] = run_experiment(
    m['trans'], 'UNet-Transformer', train_loader, val_loader, channel_weights,
    use_physics=False, epochs=EPOCHS, lr=LR, patience=PATIENCE)

# --- Same three architectures, data + physics ---
m = make_models()
results['attn_phys'] = run_experiment(
    m['attn'], 'UNet-Attention', train_loader, val_loader, channel_weights,
    use_physics=True, epochs=EPOCHS, lr=LR, patience=PATIENCE, lambda_phys=LAMBDA_PHYS)

results['fno_phys'] = run_experiment(
    m['fno'], 'UNet-FNO', train_loader, val_loader, channel_weights,
    use_physics=True, epochs=EPOCHS, lr=LR, patience=PATIENCE, lambda_phys=LAMBDA_PHYS)

results['trans_phys'] = run_experiment(
    m['trans'], 'UNet-Transformer', train_loader, val_loader, channel_weights,
    use_physics=True, epochs=EPOCHS, lr=LR, patience=PATIENCE, lambda_phys=LAMBDA_PHYS)

print("\nAll experiments done.")
for k, r in results.items():
    print(f"  {k:<12}  {r['tag']:<34}  best val = {r['best_val_loss']:.4e}  "
          f"time = {r['time']:.1f}s")


## 12. Save Checkpoints

In [ ]:
os.makedirs('checkpoints', exist_ok=True)

# Map result key -> (builder key, use_physics)
RESULT_META = {
    'base_data':  ('base',  False),
    'attn_data':  ('attn',  False),
    'fno_data':   ('fno',   False),
    'trans_data': ('trans', False),
    'attn_phys':  ('attn',  True),
    'fno_phys':   ('fno',   True),
    'trans_phys': ('trans', True),
}

for key, r in results.items():
    builder_key, use_phys = RESULT_META[key]
    path = f"checkpoints/model_{key}.pt"
    torch.save({
        'key': key,
        'tag': r['tag'],
        'builder': builder_key,
        'use_physics': use_phys,
        'state_dict': r['model'].state_dict(),
        'best_val_loss': r['best_val_loss'],
        'time': r['time'],
        'filters': FILTERS,
        'input_hw': INPUT_HW,
    }, path)
    print(f"  saved {path}")

# History for all runs
pickle.dump({k: r['history'] for k, r in results.items()},
            open('checkpoints/training_history.pkl', 'wb'))
print("  saved checkpoints/training_history.pkl")

print("\nDone. Open DeepCFD_Testing.ipynb to generate paper-style figures and tables.")


---
# Part B — Evaluation and Paper Figures

This part consumes `results` from Part A directly (no reload needed). All artifacts are written to a `figures/` folder, with both `.pdf` (paper-ready) and `.png` (preview) versions.

Figure / table -> corresponding DeepCFD paper figure:
- `fig_test_curves.pdf` → Fig. 6
- `fig_flow_sample_<idx>.pdf` → Fig. 7
- `fig_umag_p_error_<model>.pdf` → Figs. 8–9
- `fig_distribution_<model>.pdf` → Fig. 10
- `fig_relative_errors.pdf` → Fig. 11
- `table_test_errors.csv` → Table 2
- `table_inference_speedup.csv` → Table 3
- `fig_residuals.pdf` → *new*: PDE-residual quality per model

## 13. Compact matplotlib style + figures folder

In [ ]:
import matplotlib as mpl
from pathlib import Path
import csv

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 9,
    'axes.titlesize': 9,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)
print(f'Figures will be saved to {FIG_DIR.resolve()}')

## 14. Test-set Predictions and Per-sample Errors

All predictions for all 7 models are computed once and cached. Metrics are masked to the fluid region — obstacle pixels are excluded.

In [ ]:
# We work with the test split directly from memory (already created in section 8).
# All predictions are computed with obstacle-masked metrics, which is the
# physically meaningful choice.

def predict_all(model, loader):
    """Return concatenated (predictions, ground-truth, inputs) over a loader."""
    model.eval()
    preds, gts, xs = [], [], []
    with torch.no_grad():
        for bx, by in loader:
            bx = bx.to(device)
            out = model(bx).cpu()
            preds.append(out)
            gts.append(by)
            xs.append(bx.cpu())
    return torch.cat(preds), torch.cat(gts), torch.cat(xs)


def per_sample_sse(pred, gt, fluid_mask):
    """Per-sample sum of squared errors (total + per-channel), masked to fluid region.

    Returns tensor of shape [N, 4]: [total, Ux, Uy, p]. Matches the metric used
    in the original DeepCFD evaluation script.
    """
    sq = (pred - gt) ** 2
    m = fluid_mask  # [N, 1, H, W]
    me = m.expand_as(sq)
    # Sum over spatial+channel for total; sum over spatial per channel.
    total = (sq * me).sum(dim=(1, 2, 3))
    per = [(sq[:, c:c+1] * m).sum(dim=(1, 2, 3)) for c in range(3)]
    return torch.stack([total, per[0], per[1], per[2]], dim=1)


def per_sample_mse(pred, gt, fluid_mask):
    """Per-sample MSE (total + per-channel) over the fluid region. Shape [N, 4]."""
    sq = (pred - gt) ** 2
    m = fluid_mask
    me = m.expand_as(sq)
    denom_all = me.sum(dim=(1, 2, 3)).clamp_min(1.0)
    denom_ch  = m.sum(dim=(1, 2, 3)).clamp_min(1.0)
    total = (sq * me).sum(dim=(1, 2, 3)) / denom_all
    per = [((sq[:, c:c+1] * m).sum(dim=(1, 2, 3)) / denom_ch) for c in range(3)]
    return torch.stack([total, per[0], per[1], per[2]], dim=1)


# Compute predictions once per model; cache everything.
cache = {}
print("Evaluating all models on the test set ...")
for key, r in results.items():
    preds, gts, xs = predict_all(r['model'], test_loader)
    fluid = (xs[:, 1:2] > 0.5).float()
    sse  = per_sample_sse(preds, gts, fluid)
    mse  = per_sample_mse(preds, gts, fluid)
    cache[key] = dict(preds=preds, gts=gts, xs=xs, fluid=fluid, sse=sse, mse=mse)
    print(f"  {key:<12}  test MSE = {mse[:, 0].mean().item():.4e}")


## 15. Error Table (DeepCFD Table 2 equivalent)

Sum of squared errors per sample (paper convention) and MSE, per channel + total.

In [ ]:
# ---------------------------------------------------------------
# Table equivalent to Table 2 of the DeepCFD paper:
#   Per-channel and total SUM-OF-SQUARED-ERRORS on the test set,
#   averaged across samples, for each model.
# The paper reports sum-of-squared-errors per sample (not MSE).
# ---------------------------------------------------------------

ORDER = ['base_data', 'attn_data', 'fno_data', 'trans_data',
         'attn_phys', 'fno_phys', 'trans_phys']

# Formatted table
rows = []
for key in ORDER:
    if key not in cache:
        continue
    sse = cache[key]['sse']          # [N, 4]
    mse = cache[key]['mse']
    rows.append({
        'key': key,
        'tag': results[key]['tag'],
        'params': count_params(results[key]['model']),
        'sse_total': sse[:, 0].mean().item(),
        'sse_ux':    sse[:, 1].mean().item(),
        'sse_uy':    sse[:, 2].mean().item(),
        'sse_p':     sse[:, 3].mean().item(),
        'mse_total': mse[:, 0].mean().item(),
        'mse_ux':    mse[:, 1].mean().item(),
        'mse_uy':    mse[:, 2].mean().item(),
        'mse_p':     mse[:, 3].mean().item(),
        'time_s':    results[key]['time'],
    })

# Print a paper-ready markdown table (SSE metric — matches original DeepCFD code)
header = "| Model | Params | SSE total | SSE Ux | SSE Uy | SSE p | Train (s) |"
sep    = "|-------|-------:|----------:|-------:|-------:|------:|----------:|"
print(header)
print(sep)
for r in rows:
    print(f"| {r['tag']} | {r['params']:,} | "
          f"{r['sse_total']:.4f} | {r['sse_ux']:.4f} | "
          f"{r['sse_uy']:.4f} | {r['sse_p']:.4f} | {r['time_s']:.1f} |")

# Save as CSV for the paper
import csv
with open(FIG_DIR / 'table_test_errors.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
print(f"\nSaved {FIG_DIR / 'table_test_errors.csv'}")


## 16. Training Curves (DeepCFD Fig. 6 equivalent)

In [ ]:
# Figure equivalent to Fig. 6 of the paper: test (val) MSE vs epoch for
# total + per-channel breakdown.

fig, axes = plt.subplots(2, 2, figsize=(8.5, 6.0), sharex=True)
channels = [('mse', 'Total'), ('mse_ux', r'$U_x$'), ('mse_uy', r'$U_y$'), ('mse_p', r'$p$')]

# Plot only the physics-informed variants plus the baseline for readability.
to_plot = ['base_data', 'attn_phys', 'fno_phys', 'trans_phys']
colors = {'base_data': '#ff7f0e', 'attn_phys': '#1f77b4',
          'fno_phys': '#2ca02c', 'trans_phys': '#d62728'}
labels = {'base_data': 'UNet-Base (data)',
          'attn_phys': 'UNet-Attention + Phys',
          'fno_phys':  'UNet-FNO + Phys',
          'trans_phys': 'UNet-Transformer + Phys'}

for ax, (metric, title) in zip(axes.flat, channels):
    for key in to_plot:
        if key not in results:
            continue
        hist = results[key]['history']['val']
        ys = [h[metric] for h in hist]
        ax.plot(ys, label=labels.get(key, key), color=colors.get(key), linewidth=1.2)
    ax.set_ylabel(f'Test MSE — {title}')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

axes[1, 0].set_xlabel('Epoch')
axes[1, 1].set_xlabel('Epoch')
axes[0, 0].legend(loc='upper right', frameon=False)
fig.suptitle('Test error curves', y=0.99)
fig.tight_layout()
out = FIG_DIR / 'fig_test_curves.pdf'
fig.savefig(out)
fig.savefig(out.with_suffix('.png'))
plt.show()
print(f"Saved {out}")


## 17. Flow Comparison per Sample (DeepCFD Fig. 7 equivalent)

Rows: Ux / Uy / p. Columns: CFD ground-truth + the 3 physics-informed models. Obstacle pixels rendered as grey.

In [ ]:
# Figure equivalent to Fig. 7 of the paper: full velocity components + pressure
# for one sample, ground-truth vs each model in a single grid.
#
# The domain is H=172 (x, stream-wise) x W=79 (y, perpendicular).
# We display with `imshow(img.T, origin='lower')` so x is horizontal (as in the
# paper, matching the physical flow direction).

FIELDS = [('Ux', r'$U_x$ [m/s]'), ('Uy', r'$U_y$ [m/s]'), ('p', r'$p$ [Pa]')]


def obstacle_mask_from_x(x_sample):
    # x_sample: [3, H, W]
    return (x_sample[1] < 0.5).numpy()  # True where obstacle


def _masked_field(field_np, obs):
    """Return a masked array with obstacle pixels hidden (grey in the plot)."""
    return np.ma.array(field_np, mask=obs)


def fig_flow_comparison(sample_idx, models_keys, filename, title_prefix='Sample'):
    """Rows = Ux / Uy / p. Columns = GT + models."""
    n_cols = 1 + len(models_keys)
    fig, axes = plt.subplots(3, n_cols, figsize=(2.6 * n_cols, 5.5))

    gt  = test_y[sample_idx].numpy()            # [3, H, W]
    x_s = test_x[sample_idx]                    # [3, H, W]
    obs = obstacle_mask_from_x(x_s)             # [H, W]

    cmap_vel = mpl.colormaps.get_cmap('RdBu_r').copy()
    cmap_vel.set_bad('lightgray', 1.0)
    cmap_p = mpl.colormaps.get_cmap('jet').copy()
    cmap_p.set_bad('lightgray', 1.0)

    for row, (name, label) in enumerate(FIELDS):
        # Scale shared across GT + all predictions on this field
        all_data = [gt[row]]
        for k in models_keys:
            all_data.append(cache[k]['preds'][sample_idx, row].numpy())
        vmin = min(d.min() for d in all_data)
        vmax = max(d.max() for d in all_data)
        if name == 'Uy':
            v = max(abs(vmin), abs(vmax)); vmin, vmax = -v, v

        cmap = cmap_p if name == 'p' else cmap_vel

        # Column 0: GT
        ax = axes[row, 0]
        im = ax.imshow(_masked_field(gt[row], obs).T, cmap=cmap, origin='lower',
                       vmin=vmin, vmax=vmax, aspect='auto')
        ax.set_ylabel(label)
        if row == 0: ax.set_title('CFD')
        ax.set_xticks([]); ax.set_yticks([])

        # Columns 1..: predictions
        for col, k in enumerate(models_keys, start=1):
            ax = axes[row, col]
            pred = cache[k]['preds'][sample_idx, row].numpy()
            ax.imshow(_masked_field(pred, obs).T, cmap=cmap, origin='lower',
                      vmin=vmin, vmax=vmax, aspect='auto')
            if row == 0:
                tag = results[k]['tag'].replace(' (Data only)', '').replace(' + Physics', '\n+ Physics')
                ax.set_title(tag)
            ax.set_xticks([]); ax.set_yticks([])

        # One shared horizontal colorbar per row
        cbar = fig.colorbar(im, ax=axes[row, :].ravel().tolist(), location='right',
                            shrink=0.8, pad=0.01, aspect=12)

    fig.suptitle(f'{title_prefix} #{sample_idx}', y=0.995)
    out = FIG_DIR / filename
    fig.savefig(out)
    fig.savefig(out.with_suffix('.png'))
    plt.show()
    print(f"Saved {out}")


# Show 3 samples, each displaying GT + the 3 physics-informed models.
PHYS_KEYS = ['attn_phys', 'fno_phys', 'trans_phys']
for i, idx in enumerate([0, 10, 25][:min(3, len(test_x))]):
    fig_flow_comparison(idx, PHYS_KEYS, filename=f'fig_flow_sample_{idx}.pdf')


## 18. |U|, p, and Absolute Error (DeepCFD Figs. 8–9 equivalent)

Three-row layout: CFD / model / absolute error — for velocity magnitude and pressure.

In [ ]:
# Figure equivalent to Figs. 8-9 of the paper:
#   Row 1: CFD ground-truth   — |U|   and   p
#   Row 2: Model prediction   — |U|   and   p
#   Row 3: Absolute error     — |U|   and   p
#
# One figure per (sample, model). Space permitting, we loop a few combinations.

def fig_umag_p_error(sample_idx, model_key, filename):
    gt  = test_y[sample_idx].numpy()            # [3, H, W]
    x_s = test_x[sample_idx]
    obs = obstacle_mask_from_x(x_s)
    pred = cache[model_key]['preds'][sample_idx].numpy()

    umag_gt = np.sqrt(gt[0] ** 2 + gt[1] ** 2)
    umag_pr = np.sqrt(pred[0] ** 2 + pred[1] ** 2)
    p_gt = gt[2]
    p_pr = pred[2]

    err_umag = np.abs(umag_pr - umag_gt)
    err_p    = np.abs(p_pr - p_gt)

    fig, axes = plt.subplots(3, 2, figsize=(6.4, 5.4))

    cmap_jet = mpl.colormaps.get_cmap('jet').copy()
    cmap_jet.set_bad('lightgray', 1.0)
    cmap_err = mpl.colormaps.get_cmap('hot').copy()
    cmap_err.set_bad('lightgray', 1.0)

    row_labels = ['CFD', results[model_key]['tag'].split(' (')[0].split(' +')[0], '|Error|']

    # Column scales
    umag_max = max(umag_gt.max(), umag_pr.max())
    p_vmin = min(p_gt.min(), p_pr.min())
    p_vmax = max(p_gt.max(), p_pr.max())

    def _plot(ax, img, cmap, vmin=None, vmax=None, label=None):
        m = np.ma.array(img, mask=obs)
        im = ax.imshow(m.T, cmap=cmap, origin='lower', vmin=vmin, vmax=vmax, aspect='auto')
        ax.set_xticks([]); ax.set_yticks([])
        return im

    # Row 0 — ground truth
    im_u0 = _plot(axes[0, 0], umag_gt, cmap_jet, 0, umag_max)
    im_p0 = _plot(axes[0, 1], p_gt,    cmap_jet, p_vmin, p_vmax)
    # Row 1 — prediction
    _plot(axes[1, 0], umag_pr, cmap_jet, 0, umag_max)
    _plot(axes[1, 1], p_pr,    cmap_jet, p_vmin, p_vmax)
    # Row 2 — absolute error (percentile to avoid outliers blowing out the scale)
    err_u_max = np.percentile(err_umag[~obs], 99)
    err_p_max = np.percentile(err_p[~obs], 99)
    im_u2 = _plot(axes[2, 0], err_umag, cmap_err, 0, err_u_max)
    im_p2 = _plot(axes[2, 1], err_p,    cmap_err, 0, err_p_max)

    for i, lbl in enumerate(row_labels):
        axes[i, 0].set_ylabel(lbl)
    axes[0, 0].set_title(r'$|\mathbf{U}|$ [m/s]')
    axes[0, 1].set_title(r'$p$ [Pa]')

    # Compact colorbars — one per row on the right of each column
    fig.colorbar(im_u0, ax=axes[0:2, 0].ravel().tolist(),
                 location='right', shrink=0.7, pad=0.02, aspect=12)
    fig.colorbar(im_p0, ax=axes[0:2, 1].ravel().tolist(),
                 location='right', shrink=0.7, pad=0.02, aspect=12)
    fig.colorbar(im_u2, ax=[axes[2, 0]],
                 location='right', shrink=0.9, pad=0.02, aspect=8)
    fig.colorbar(im_p2, ax=[axes[2, 1]],
                 location='right', shrink=0.9, pad=0.02, aspect=8)

    fig.suptitle(f"{results[model_key]['tag']}  —  sample #{sample_idx}", y=0.995)
    out = FIG_DIR / filename
    fig.savefig(out)
    fig.savefig(out.with_suffix('.png'))
    plt.show()
    print(f"Saved {out}")


# Loop: for each of the 3 physics-informed models, one representative sample.
for m_key, idx in zip(['attn_phys', 'fno_phys', 'trans_phys'], [0, 10, 25]):
    if idx >= len(test_x):
        idx = 0
    fig_umag_p_error(idx, m_key, filename=f'fig_umag_p_error_{m_key}_sample{idx}.pdf')


## 19. Data Distributions (DeepCFD Fig. 10 equivalent)

Histograms of ground-truth vs predicted values on the test set, per variable and combined. Means and standard deviations annotated in-panel.

In [ ]:
# Figure equivalent to Fig. 10 of the paper: ground-truth vs predicted data
# distribution over the full test set (all variables + combined).
#
# Only fluid-region pixels enter the histograms.

def collect_fluid_values(arr4d, mask4d, channel=None):
    """arr4d, mask4d: [N, C, H, W]; returns flat numpy array for pixels in fluid."""
    if channel is not None:
        a = arr4d[:, channel:channel+1]
        m = mask4d
    else:
        a = arr4d
        m = mask4d.expand_as(arr4d)
    return a[m.bool()].numpy()


def fig_distribution(model_key, filename):
    ck = cache[model_key]
    preds = ck['preds']; gts = ck['gts']; m = ck['fluid']

    fig, axes = plt.subplots(2, 2, figsize=(8.5, 5.5))
    axes = axes.flat

    panels = [
        (None,          'Total',     axes[0]),
        (0,             r'$U_x$',    axes[1]),
        (1,             r'$U_y$',    axes[2]),
        (2,             r'$p$',      axes[3]),
    ]

    for ch, title, ax in panels:
        a_gt = collect_fluid_values(gts,   m, ch)
        a_pr = collect_fluid_values(preds, m, ch)
        lo = np.percentile(np.concatenate([a_gt, a_pr]), 0.5)
        hi = np.percentile(np.concatenate([a_gt, a_pr]), 99.5)
        bins = np.linspace(lo, hi, 80)
        ax.hist(a_gt, bins=bins, alpha=0.55, color='#d62728', label='simpleFOAM')
        ax.hist(a_pr, bins=bins, alpha=0.55, color='#1f77b4', label=results[model_key]['tag'])
        ax.set_title(title)
        ax.set_xlabel('Value'); ax.set_ylabel('Count')
        stats = (f"GT: $\\mu$={a_gt.mean():.4f}, $\\sigma$={a_gt.std():.4f}\n"
                 f"Pred: $\\mu$={a_pr.mean():.4f}, $\\sigma$={a_pr.std():.4f}")
        ax.text(0.98, 0.95, stats, transform=ax.transAxes, ha='right', va='top',
                fontsize=7, bbox=dict(facecolor='white', alpha=0.8, edgecolor='0.7'))

    axes[0].legend(loc='upper left', frameon=False)
    fig.suptitle(f"Data distribution — {results[model_key]['tag']}", y=0.99)
    fig.tight_layout()
    out = FIG_DIR / filename
    fig.savefig(out); fig.savefig(out.with_suffix('.png'))
    plt.show()
    print(f"Saved {out}")


for key in ['attn_phys', 'fno_phys', 'trans_phys']:
    fig_distribution(key, filename=f'fig_distribution_{key}.pdf')


## 20. Relative Error Distributions (DeepCFD Fig. 11 equivalent)

Relative error $|p - \mathrm{gt}| / (|\mathrm{gt}| + k) \times 100\%$ with $k = 10^{-4}$ (paper convention).

In [ ]:
# Figure equivalent to Fig. 11 of the paper: relative error distributions
# for each variable (total + per channel), across all test samples.
#
# Relative error = |pred - gt| / (|gt| + k) * 100%, with small k to avoid
# division by zero (paper uses k = 1e-4).

K = 1e-4


def relative_errors(pred, gt, mask, channel=None):
    if channel is not None:
        p = pred[:, channel:channel+1]
        g = gt[:, channel:channel+1]
        m = mask
    else:
        p = pred; g = gt; m = mask.expand_as(pred)
    rel = (p - g).abs() / (g.abs() + K) * 100.0
    return rel[m.bool()].numpy()


def fig_relative_errors(models_keys, filename, colors=None, clip_pct=100):
    if colors is None:
        colors = ['#ff7f0e', '#1f77b4', '#2ca02c', '#d62728']

    fig, axes = plt.subplots(2, 2, figsize=(8.5, 5.5))
    axes = axes.flat

    bins = np.linspace(0, clip_pct, 80)

    panels = [
        (None, 'Total',   axes[0]),
        (0,    r'$U_x$',  axes[1]),
        (1,    r'$U_y$',  axes[2]),
        (2,    r'$p$',    axes[3]),
    ]

    for ch, title, ax in panels:
        for col_idx, key in enumerate(models_keys):
            rel = relative_errors(cache[key]['preds'], cache[key]['gts'],
                                  cache[key]['fluid'], channel=ch)
            rel = np.clip(rel, 0, clip_pct)
            ax.hist(rel, bins=bins, alpha=0.5, color=colors[col_idx % len(colors)],
                    label=results[key]['tag'])
        ax.set_title(title)
        ax.set_xlabel('Relative error [%]')
        ax.set_ylabel('Count')
        ax.grid(True, alpha=0.3)

    axes[0].legend(loc='upper right', frameon=False, fontsize=7)
    fig.suptitle('Relative error distributions (test set)', y=0.99)
    fig.tight_layout()
    out = FIG_DIR / filename
    fig.savefig(out); fig.savefig(out.with_suffix('.png'))
    plt.show()
    print(f"Saved {out}")


# One figure: baseline + the 3 physics-informed models
fig_relative_errors(
    ['base_data', 'attn_phys', 'fno_phys', 'trans_phys'],
    filename='fig_relative_errors.pdf',
)


## 21. Inference Speedup (DeepCFD Table 3 equivalent)

Inference time per batch size and speedup over the CFD reference. The paper reports 52.51 s/sample for single-core `simpleFoam` on a Xeon E-2146G; update `CFD_TIME_S` with your measured value if you want a speedup relative to your own OpenFOAM timing.

In [ ]:
# Table equivalent to Table 3 of the paper: inference time + speedup vs CFD.
#
# We time each model on the actual test device, with batch sizes 1, 10, 100.
# For the CFD reference we use the paper's reported average of 52.51 s/sample
# (single-core simpleFoam on an Intel Xeon E-2146G); users can override if they
# have measured their own CFD timing.

CFD_TIME_S = 52.51  # paper value; replace with your measured simpleFoam time

BATCH_SIZES = [1, 10, 100]
N_RUNS = 50  # reduce/increase as desired

# Use the first sample(s) of the test set, repeated to form the requested batch
x_single = test_x[0:1].to(device)

def time_inference(model, batch_size, n_runs=N_RUNS):
    """Average + std inference time (seconds) for `batch_size` samples."""
    model.eval()
    x_batch = x_single.repeat(batch_size, 1, 1, 1)
    # warm-up
    with torch.no_grad():
        for _ in range(3):
            _ = model(x_batch)
    if device.type == 'cuda':
        torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(x_batch)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    t = np.array(times)
    return float(t.mean()), float(t.std())


rows_time = []
print(f"\nReference CFD time (paper, single-core CPU): {CFD_TIME_S:.2f} s/sample")
print(f"Measuring inference on {device} (batches of 1, 10, 100) ...\n")

for key in ORDER:
    if key not in results:
        continue
    m = results[key]['model']
    row = {'key': key, 'tag': results[key]['tag'], 'params': count_params(m)}
    for bs in BATCH_SIZES:
        mu, sd = time_inference(m, bs)
        per_sample = mu / bs
        speedup = CFD_TIME_S / per_sample
        row[f'time_bs{bs}_s']    = mu
        row[f'time_bs{bs}_std']  = sd
        row[f'per_sample_bs{bs}'] = per_sample
        row[f'speedup_bs{bs}']    = speedup
    rows_time.append(row)

# Pretty print
print(f"{'Model':<32}  {'BS=1 (s)':>10}  {'Sp. BS=1':>10}  "
      f"{'BS=10 (s)':>11}  {'Sp. BS=10':>10}  "
      f"{'BS=100 (s)':>12}  {'Sp. BS=100':>11}")
print('-' * 105)
for r in rows_time:
    print(f"{r['tag']:<32}  "
          f"{r['time_bs1_s']:>10.3e}  {r['speedup_bs1']:>10.2e}  "
          f"{r['time_bs10_s']:>11.3e}  {r['speedup_bs10']:>10.2e}  "
          f"{r['time_bs100_s']:>12.3e}  {r['speedup_bs100']:>11.2e}")

# CSV export
with open(FIG_DIR / 'table_inference_speedup.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows_time[0].keys()))
    w.writeheader(); w.writerows(rows_time)
print(f"\nSaved {FIG_DIR / 'table_inference_speedup.csv'}")


## 22. Physics Residuals Comparison (extra)

Point-wise continuity and momentum residuals on a test sample, one row per model. Low residuals → predictions satisfy the governing equations, not just the data. RMS values reported in-panel.

In [ ]:
# Extra figure: PDE residuals (continuity + momentum) on a test sample.
# Quantifies how well each model satisfies the Navier-Stokes equations,
# regardless of pointwise accuracy. Useful for paper discussion.

phys = PhysicsInformedLoss().to(device)


def compute_residuals(model, x_sample):
    """Compute continuity and momentum residuals for one sample, as [H, W] numpy arrays."""
    model.eval()
    with torch.no_grad():
        x_in = x_sample.unsqueeze(0).to(device)
        y = model(x_in)
        ux, uy, p = y[:, 0:1], y[:, 1:2], y[:, 2:3]
        m = phys.region_masks(x_in)['interior']

        div = (phys.ddx(ux) + phys.ddy(uy)) * m
        res_x = (ux * phys.ddx(ux) + uy * phys.ddy(ux) + phys.ddx(p)
                 - phys.nu * (phys.d2dx2(ux) + phys.d2dy2(ux))) * m
        res_y = (ux * phys.ddx(uy) + uy * phys.ddy(uy) + phys.ddy(p)
                 - phys.nu * (phys.d2dx2(uy) + phys.d2dy2(uy))) * m
    obs = (x_sample[1] < 0.5).numpy()
    return (div[0, 0].cpu().numpy(), res_x[0, 0].cpu().numpy(),
            res_y[0, 0].cpu().numpy(), obs)


def fig_residuals(sample_idx, models_keys, filename):
    fig, axes = plt.subplots(len(models_keys), 3,
                             figsize=(9.0, 2.2 * len(models_keys)))
    if len(models_keys) == 1:
        axes = axes[None, :]

    cmap = mpl.colormaps.get_cmap('RdBu_r').copy()
    cmap.set_bad('lightgray', 1.0)

    col_titles = [r'$\nabla \cdot \mathbf{u}$', 'Mom$_x$', 'Mom$_y$']

    for row, key in enumerate(models_keys):
        div, rx, ry, obs = compute_residuals(results[key]['model'], test_x[sample_idx])
        data = [div, rx, ry]
        for col in range(3):
            d = data[col]
            vmax = np.percentile(np.abs(d[~obs]), 97)
            vmax = max(vmax, 1e-12)
            m = np.ma.array(d, mask=obs)
            im = axes[row, col].imshow(m.T, cmap=cmap, origin='lower',
                                       vmin=-vmax, vmax=vmax, aspect='auto')
            axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
            if row == 0:
                axes[row, col].set_title(col_titles[col])
            rms = float(np.sqrt(np.mean(d[~obs] ** 2)))
            axes[row, col].text(0.02, 0.95, f'RMS={rms:.2e}', transform=axes[row, col].transAxes,
                                va='top', fontsize=7,
                                bbox=dict(facecolor='white', alpha=0.8, edgecolor='0.7'))
            if col == 2:
                fig.colorbar(im, ax=axes[row, col], shrink=0.9, pad=0.02)
        axes[row, 0].set_ylabel(results[key]['tag'].split(' (')[0].split(' +')[0]
                                + ('\n+Phys' if results[key]['use_physics'] else ''))

    fig.suptitle(f'Physics residuals — sample #{sample_idx}', y=0.995)
    fig.tight_layout()
    out = FIG_DIR / filename
    fig.savefig(out); fig.savefig(out.with_suffix('.png'))
    plt.show()
    print(f"Saved {out}")


fig_residuals(0, ['base_data', 'attn_phys', 'fno_phys', 'trans_phys'],
              filename='fig_residuals.pdf')


## 23. Summary

In [ ]:
print("=" * 72)
print("Artifacts generated (folder: figures/)")
print("=" * 72)
for p in sorted(FIG_DIR.glob('*')):
    size = p.stat().st_size / 1024
    print(f"  {p.name:<45}  {size:>8.1f} KB")

print()
print("Figures for the paper")
print("  fig_test_curves.pdf               -> like Fig. 6 of DeepCFD")
print("  fig_flow_sample_<idx>.pdf         -> like Fig. 7 of DeepCFD")
print("  fig_umag_p_error_<model>.pdf      -> like Figs. 8-9 of DeepCFD")
print("  fig_distribution_<model>.pdf      -> like Fig. 10 of DeepCFD")
print("  fig_relative_errors.pdf           -> like Fig. 11 of DeepCFD")
print("  fig_residuals.pdf                 -> extra: PDE residual quality")
print()
print("Tables for the paper")
print("  table_test_errors.csv             -> like Table 2 of DeepCFD")
print("  table_inference_speedup.csv       -> like Table 3 of DeepCFD")
print()
print("Checkpoints")
print("  checkpoints/model_<key>.pt        -> one per (architecture, mode)")
print("  checkpoints/training_history.pkl  -> full loss/metric history")
